# Hands-on RAG for Education
## Notebook 1: Three ready-made ways to build a RAG assistant

ACM SIGCITE 2026 tutorial · Yong Zheng, Illinois Institute of Technology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/academictutorials/academictutorials.github.io/blob/main/EdRAG/resources/materials/practice_offline/01_ready_made_rag_colab.ipynb)

**Offline version** for Google Colab. Use it after the tutorial: you need only a Google account and your own free Gemini API key.

In this notebook you try three of the four approaches from the tutorial on the same documents and the same five test questions, and you score every answer yourself:

| Approach | Tool | What you do here |
| --- | --- | --- |
| 1. Consumer platform | NotebookLM | Build your own NotebookLM notebook in the browser and ask the questions |
| 2. Managed RAG API | Gemini File Search | Use your own free Gemini API key to build a document store and ask the questions in a few lines of code |
| 3. Self-hosted application | AnythingLLM | Read and score the answers recorded on the tutorial server, or install AnythingLLM Desktop yourself |

Your scores are saved to a small file. Notebook 2, where you build the fourth approach from scratch, reads that file and compares all four approaches.

**How to run:** click a cell and press **Shift + Enter**. Run the cells from top to bottom.

## 0. Setup

This cell prepares Colab. It installs the one package Colab does not have (`google-genai`, about 30 seconds) and downloads the two lab documents. Run it every time you open the notebook, because Colab starts each session on a fresh machine.

In [ ]:
import os, sys, json, pathlib, urllib.request, importlib

import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==2.30.1"], check=True)

import httpx
from IPython.display import Markdown, display
from google import genai

# Helper code shared by both notebooks: downloads, scores and charts, and recorded answers
try:
    urllib.request.urlretrieve("https://academictutorials.github.io/EdRAG/resources/materials/practice_offline/edrag_utils.py",
                               "edrag_utils.py")
except Exception as exc:
    print("Could not refresh edrag_utils.py, using the copy already here:", exc)
import edrag_utils
importlib.reload(edrag_utils)
from edrag_utils import *

DOCUMENTS = {"ITM_Student_Handbook.pdf": "", "Tuition_Fees.html": ""}   # the two lab documents
download(DOCUMENTS)

USER = "colab"

## 1. The five test questions and the scoring rubric

Expected answers, sources, and common mistakes for every question are on the tutorial website:
**https://academictutorials.github.io/EdRAG/edrag-test-questions.html**

Q1–Q4 have answers in the documents. **Q5 does not**: it checks whether an approach says so instead of guessing.

**Give each answer one overall correctness score:**

* **0 = incorrect**: the answer contains any wrong or invented information, even if the rest is right
* **1 = partially correct**: everything stated is correct, but some expected information is missing
* **2 = completely correct**: correct and complete

Use the five criteria below as a checklist when you decide. Later, you can also score each criterion separately with the same 0/1/2 scale (optional cells marked *Detailed scoring*).

| Criterion | What to check |
| --- | --- |
| Retrieval relevance | The retrieved passages are the ones that hold the answer (for Q5, the closest related passages). If a tool does not show what it retrieved, judge from its citations. |
| Grounding | Every statement in the answer is supported by the retrieved passages. |
| Citation accuracy | The cited document and page point to the passage the answer actually uses. |
| Absence of unsupported claims | The answer adds no facts, numbers, or advice that the documents do not contain. |
| Correct refusal | When the documents do not hold the answer, or hold only part of it, the answer says so. When they do hold it, the answer gives it. |

* Overall correctness: up to 2 points per answer, **10 points** per approach for Q1–Q5.
* Scored by criterion: up to 10 points per answer, **50 points** per approach.

In [ ]:
# The five questions (defined in edrag_utils.py, shared with Notebook 2)
for q, text in QUESTIONS.items():
    print(f"{q}: {text}")

Your scores are handled by four functions from `edrag_utils.py`:

* `save_overall(approach, scores)`: one overall correctness score (0, 1, or 2) per question.
* `save_detailed(approach, scores)`: five criterion scores per question (optional).
* `compare_overall()` and `compare_detailed()`: tables and charts for every approach you have scored so far.

Scores are saved in `my_scores.json`, so Notebook 2 can add the approach you build yourself.

---
## 2. Approach 1: NotebookLM (consumer platform)

A consumer platform needs no code at all. Build your own NotebookLM notebook with the two lab documents (about 3 minutes):

1. Download the handbook PDF to your computer:
   **https://academictutorials.github.io/EdRAG/resources/materials/ITM_Student_Handbook.pdf**
2. Open **https://notebooklm.google.com** and sign in. A personal Gmail account works best; some school accounts have NotebookLM turned off.
3. Click **+ New notebook**.
4. Add the first source: **upload** the handbook PDF.
5. Add the second source: choose **Website** and paste
   **https://academictutorials.github.io/EdRAG/resources/materials/Tuition_Fees.html**
6. When both sources show as ready, ask Q1–Q5 one at a time (copy them from the list above).
7. Compare each answer with the expected answers on the website, then fill in your scores below.

Notice what you **cannot** see or change: how the documents were split, which passages were retrieved, and which model answered.

In [ ]:
# Your overall correctness score for NotebookLM, one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
notebooklm_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("notebooklm", notebooklm_scores_overall)

*Detailed scoring (optional):* score each criterion separately for NotebookLM.

In [ ]:
# Five criterion scores per question, each 0, 1, or 2
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
notebooklm_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("notebooklm", notebooklm_scores)

---
## 3. Approach 2: Gemini File Search (managed RAG API)

With a managed RAG API, you upload documents and the provider splits them, embeds them, stores them, and retrieves from them for you. You only write a few lines of code.

### 3.1 Get your own free Gemini API key (about 2 minutes)

1. Open **https://aistudio.google.com/apikey** and sign in. A personal Gmail account works best; some school accounts block AI Studio.
2. Click **Create API key** and copy it.
3. Store it in Colab so you do not have to paste it every time: click the **key icon** in Colab's left sidebar (Secrets), add a secret named `GEMINI_API_KEY` with your key as the value, and turn on **Notebook access**. If you skip this, the next cell asks you to paste the key instead.

> On the free tier, Google may use what you send to improve its products. The two lab documents are public pages, but do not type personal information into your questions.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
    print("Using the GEMINI_API_KEY secret from Colab.")
except Exception:
    GEMINI_KEY = getpass("Paste your Gemini API key: ").strip()

FS_MODEL = "gemini-3.5-flash-lite"   # free tier: 15 requests per minute
client = genai.Client(api_key=GEMINI_KEY)
USE_OWN_KEY = True
print("Model:", FS_MODEL)

### 3.2 Build your document store

These few lines are the whole “build” step of a managed RAG API: create a store and upload two files. Gemini splits, embeds, and indexes them for you. This takes about 20 seconds. Running the cell again reuses the store instead of uploading twice.

In [ ]:
import time

if USE_OWN_KEY:
    store_name = f"edrag-{USER}"
    store = next((s for s in client.file_search_stores.list() if s.display_name == store_name), None)
    if store is None:
        store = client.file_search_stores.create(config={"display_name": store_name})
        for file_name in DOCUMENTS:
            t0 = time.time()
            op = client.file_search_stores.upload_to_file_search_store(
                file=str(DATA / file_name), file_search_store_name=store.name,
                config={"display_name": file_name})
            while not op.done:
                time.sleep(3)
                op = client.operations.get(op)
            print(f"Indexed {file_name} in {time.time() - t0:.0f} s")
    else:
        print("Reusing your store")
    for doc in client.file_search_stores.documents.list(parent=store.name):
        print(" ", doc.display_name, doc.state)
else:
    print("Skipped: using the instructor's store.")

### 3.3 Ask the five questions

`ask_file_search()` sends one question with the File Search tool turned on, then prints the answer and the pages it cites. Answers are displayed through `show_answer()` (in `edrag_utils.py`), which protects dollar signs: otherwise Jupyter would treat the text between two `$` signs as a math formula and swallow it.

In [ ]:
def ask_file_search(question):
    # Ask one question with the File Search tool turned on; print the answer and the pages it cites
    if USE_OWN_KEY:
        result = client.interactions.create(
            model=FS_MODEL, input=question,
            tools=[{"type": "file_search", "file_search_store_names": [store.name]}])
        answer, cites = "", set()
        for step in result.steps:
            if step.type == "model_output":
                for block in step.content:
                    if block.type == "text":
                        answer += block.text
                        for a in block.annotations or []:
                            if a.type == "file_citation":
                                cites.add((a.file_name, getattr(a, "page_number", None)))
    show_answer(answer)                      # dollar signs protected from math rendering
    print("Cited:", ", ".join(f"{f} p.{p}" if p else f for f, p in sorted(cites, key=str)) or "nothing")


for q, text in QUESTIONS.items():
    display(Markdown(f"---\n### {q}: {text}"))
    ask_file_search(text)

In [ ]:
# Your overall correctness score for Gemini File Search, one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
file_search_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("file_search", file_search_scores_overall)

*Detailed scoring (optional):* score each criterion separately for Gemini File Search.

In [ ]:
# Five criterion scores per question, each 0, 1, or 2
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
file_search_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("file_search", file_search_scores)

**Optional.** Your store stays in your Google project until you delete it. Run this cell when you are done.

In [ ]:
if USE_OWN_KEY:
    client.file_search_stores.delete(name=store.name, config={"force": True})
    print("Deleted", store.name)

---
## 4. Approach 3: AnythingLLM (self-hosted application)

During the tutorial, AnythingLLM ran on the tutorial server with two workspaces that used the **same model** (Qwen3.8-Flash):

| Workspace | Documents | Chat mode |
| --- | --- | --- |
| `norag` (Tutorial-NoRAG) | none | Chat: the model answers from memory |
| `edrag` (Tutorial-EdRAG) | the two lab documents | Query: answers only from the documents, refuses when they have nothing |

That server is not available after the tutorial. You have two options:

* **4.1–4.2: read the recorded answers** from the tutorial server and score them (no setup).
* **4.3 (optional): install AnythingLLM Desktop** on your own computer and ask the questions yourself.

### 4.1 Recorded answers: same model, with and without documents

In [ ]:
RECORDED = load_recorded()          # answers recorded on the tutorial server (edrag_utils.py)
print("Recorded on", RECORDED["recorded"], "|", RECORDED["model"])


def show_recorded(item):
    # Print one recorded answer, then the passages AnythingLLM retrieved for it
    show_answer(item["answer"])              # dollar signs protected from math rendering
    print(f"[{item['workspace']} | {item['mode']} mode | {len(item['sources'])} passages retrieved]")
    for s in item["sources"]:
        print(f"  - {s['title']}: {s['excerpt']}")


for item in RECORDED["with_and_without_documents"]:
    display(Markdown(f"---\n### Workspace: {item['workspace']}\n**Question:** {item['question']}"))
    show_recorded(item)

Compare the two answers with the expected answer for Q3 on the website: **OTS Support Desk, supportdesk@illinoistech.edu, 312.567.DESK (3375)**, and ITM faculty, teaching assistants, and staff cannot help with Canvas.

### 4.2 Recorded answers to the five questions (workspace `edrag`)

Look at the retrieved passages too: the tuition page was uploaded as a raw HTML file, and AnythingLLM split the page's HTML code, scripts, and menus into chunks.

In [ ]:
for q, item in RECORDED["edrag"].items():
    display(Markdown(f"---\n### {q}: {QUESTIONS[q]}"))
    show_recorded(item)

In [ ]:
# Your overall correctness score for AnythingLLM (workspace edrag), one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
anythingllm_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("anythingllm", anythingllm_scores_overall)

*Detailed scoring (optional):* score each criterion separately for AnythingLLM (workspace edrag).

In [ ]:
# Five criterion scores per question, each 0, 1, or 2
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
anythingllm_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("anythingllm", anythingllm_scores)

### 4.3 Optional: run AnythingLLM yourself (about 15 minutes)

AnythingLLM Desktop runs on Windows, macOS, and Linux. Your documents stay on your computer; only your questions and the retrieved passages go to the language model you choose.

1. Download and install **AnythingLLM Desktop** from **https://anythingllm.com/desktop**.
2. On first launch, choose **Gemini** as the LLM provider, paste your Gemini API key, and pick `gemini-3.5-flash-lite`. Keep the default embedder and vector database.
3. Create a workspace, upload `ITM_Student_Handbook.pdf` and `Tuition_Fees.html` (from this notebook's `data` folder or the tutorial website), and click **Save and Embed**.
4. In the workspace settings, set the chat mode to **Query**.
5. Ask Q1–Q5 in the app and compare with the recorded answers above. Did a different model or embedder change the answers to Q1 and Q2?

Colab cannot reach an app on your own computer, so you ask the questions in the AnythingLLM window rather than in this notebook.

---
## 5. Compare the three approaches

Run this cell after you have saved overall scores for at least one approach. Notebook 2 adds the approach you build yourself.

In [ ]:
compare_overall()

# Colab deletes files when the session ends: download your scores to keep them
download_scores()

*Detailed scoring (optional):* compare the scores by criterion.

In [ ]:
compare_detailed()

**Questions to think about**

* Which approach scored best overall, and on which question did the approaches differ most?
* Which approach let you see what it retrieved? How did that affect your scoring?
* Which approach would you trust with your own institution's documents, and why?

Your scores were downloaded as `my_scores.json`. In Notebook 2, upload it with the Files panel (folder icon on the left) before you compare.